# Jz_2_Homodyne_N2 - Notebook Debug Version

Notebook allineato a `Jz_2_Homodyne_N2.py`, ma orientato al debug numerico.

Questo notebook usa di default una configurazione **legacy-comparable** (`eta_1 = eta_2`) per confrontare i risultati con i vecchi file `.npz`/`.json`.

In [ ]:
import json
import os
from fractions import Fraction
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib

def _in_notebook() -> bool:
    try:
        from IPython import get_ipython
        shell = get_ipython()
        return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False

if not _in_notebook():
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qutip import (
    basis,
    expect,
    ket2dm,
    qeye,
    sigmax,
    sigmay,
    sigmaz,
    smesolve,
    tensor,
    variance,
)

In [ ]:
# Basis states and collective operators for N=2
gnd = basis(2, 0)
exc = basis(2, 1)

plus_state = (gnd + exc).unit()
PlusPlus = tensor(plus_state, plus_state)
ee = tensor(exc, exc)

I_2 = qeye(2)
J_x = 0.5 * (tensor(sigmax(), I_2) + tensor(I_2, sigmax()))
J_y = 0.5 * (tensor(sigmay(), I_2) + tensor(I_2, sigmay()))
J_z = 0.5 * (tensor(sigmaz(), I_2) + tensor(I_2, sigmaz()))

OMEGA = 0.0
H_free = 0.5 * OMEGA * (tensor(sigmaz(), I_2) + tensor(I_2, sigmaz()))

Cp = np.sqrt(0.5) * (tensor(sigmaz(), I_2) + tensor(I_2, sigmaz()))
Cm = np.sqrt(0.5) * (tensor(sigmaz(), I_2) - tensor(I_2, sigmaz()))

In [ ]:
def css_2(theta: float, phi: float):
    single_qubit = (
        np.sin(theta / 2.0) * basis(2, 0)
        + np.exp(1j * phi) * np.cos(theta / 2.0) * basis(2, 1)
    )
    return tensor(single_qubit, single_qubit)


def collapsing_operators(gamma: float, phi_1: float, phi_2: float, eta_1: float, eta_2: float):
    return [
        np.sqrt(gamma * eta_1) * np.exp(1j * phi_1) * Cp,
        np.sqrt(gamma * eta_2) * np.exp(1j * phi_2) * Cm,
    ]


def Variance_z(t, state):
    return variance(J_z, state)


def concurrence_pure_state(state):
    c = state.full().flatten()
    return 2.0 * np.abs(c[0] * c[3] - c[1] * c[2])


def concurrence_for_solver_general(t, state):
    if state.isket:
        conc = concurrence_pure_state(state)
    else:
        sysy = tensor(sigmay(), sigmay())
        rho_tilde = (state * sysy) * (state.conj() * sysy)
        evals = rho_tilde.eigenenergies()
        evals = abs(np.sort(np.real(evals)))
        sqrt_evals = np.sqrt(evals)
        lsum = sqrt_evals[3] - sqrt_evals[2] - sqrt_evals[1] - sqrt_evals[0]
        conc = np.maximum(0.0, lsum)
    return float(np.real_if_close(conc))


def eval_spin_components(state):
    return expect(J_x, state), expect(J_y, state), expect(J_z, state)


def xi_KU_solver(t, rho, N=None, tol=1e-12):
    if N is None:
        dim = rho.shape[0]
        N = int(round(np.log2(dim)))

    Jx_exp, Jy_exp, Jz_exp = eval_spin_components(rho)
    Jmean = np.array(
        [
            float(np.real_if_close(Jx_exp)),
            float(np.real_if_close(Jy_exp)),
            float(np.real_if_close(Jz_exp)),
        ]
    )
    m = np.linalg.norm(Jmean)

    Js = [J_x, J_y, J_z]
    C = np.zeros((3, 3), dtype=float)
    for i, Ji in enumerate(Js):
        for j, Jj in enumerate(Js):
            second_moment = 0.5 * expect(Ji * Jj + Jj * Ji, rho)
            C[i, j] = float(np.real_if_close(second_moment - Jmean[i] * Jmean[j]))

    if m > tol:
        u = Jmean / m
        a = np.array([1.0, 0.0, 0.0]) if abs(u[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
        e1 = a - u * np.dot(u, a)
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(u, e1)

        C2 = np.array(
            [
                [e1 @ C @ e1, e1 @ C @ e2],
                [e2 @ C @ e1, e2 @ C @ e2],
            ],
            dtype=float,
        )
        lam_min = np.linalg.eigvalsh(C2)[0]
    else:
        lam_min = np.linalg.eigvalsh(C)[0]

    lam_min = max(float(np.real_if_close(lam_min)), 0.0)
    xi2 = (4.0 / N) * lam_min
    return float(xi2)


def norm_J(t, rho):
    return np.linalg.norm(eval_spin_components(rho))


def eta_pair_key(eta_1: float, eta_2: float) -> str:
    return f"{eta_1:g}_{eta_2:g}"


def parse_etas(raw_1: str, raw_2: str) -> list[tuple[float, float]]:
    etas_1 = [float(x.strip()) for x in raw_1.split(",") if x.strip()]
    etas_2 = [float(x.strip()) for x in raw_2.split(",") if x.strip()]
    if not etas_1 or not etas_2:
        raise ValueError("Eta list is empty. Use --etas_1 and --etas_2 with comma-separated values.")

    if len(etas_1) == 1 and len(etas_2) > 1:
        etas_1 = etas_1 * len(etas_2)
    elif len(etas_2) == 1 and len(etas_1) > 1:
        etas_2 = etas_2 * len(etas_1)
    elif len(etas_1) != len(etas_2):
        raise ValueError(
            f"Length mismatch: len(etas_1)={len(etas_1)} and len(etas_2)={len(etas_2)}. "
            "Use equal lengths, or one list with a single value to broadcast."
        )

    for eta_1, eta_2 in zip(etas_1, etas_2):
        if eta_1 < 0.0 or eta_1 > 1.0 or eta_2 < 0.0 or eta_2 > 1.0:
            raise ValueError(f"Invalid etas=({eta_1}, {eta_2}). Each eta must be in [0, 1].")
    return list(zip(etas_1, etas_2))


def build_initial_state(kind: str, theta: float, phi_state: float):
    if kind == "plusplus":
        return PlusPlus, np.pi / 2.0
    if kind == "ee":
        return ee, np.pi
    if kind == "css":
        return css_2(theta, phi_state), theta
    raise ValueError(f"Unknown state kind: {kind}")


def theoretical_concurrence_curve(times: np.ndarray, gamma: float, eta: float, theta: float) -> np.ndarray:
    return (np.sin(theta) ** 2) * np.exp(-(1.0 - eta) * times) * (1.0 - np.exp(-gamma * eta * times / 2.0))


def angle_to_path(phi, max_den=12, tol=1e-10):
    twopi = 2 * np.pi
    x = (phi % twopi + twopi) % twopi
    if abs(x) < tol:
        return "0"
    q = Fraction(x / np.pi).limit_denominator(max_den)
    n, d = q.numerator, q.denominator
    if n == 0:
        return "0"
    if d == 1:
        return "pi" if n == 1 else f"{n}pi"
    return ("pi_" + str(d)) if n == 1 else f"{n}pi_{d}"


DEFAULT_EOPS = [
    ("Conc", concurrence_for_solver_general),
    ("Xi2_KU", xi_KU_solver),
    ("Norm_J", norm_J),
]

LABELS = {
    "Conc": r"$\overline{\mathcal{C}}$",
    "Xi2_KU": r"$\overline{\xi^2_{KU}}$",
    "Norm_J": r"$\overline{|\langle \mathbf{J} \rangle|}$",
}

In [ ]:
def run_homodyne(
    rho0,
    times: np.ndarray,
    gamma: float,
    phi1: float,
    phi2: float,
    eta_1: float,
    eta_2: float,
    e_ops: list,
    ntraj: int,
    num_cpus: int,
    seed: int | None = None,
):
    solver_cpus = max(1, int(num_cpus))
    options = {
        "keep_runs_results": False,
        "num_cpus": solver_cpus,
        "map": "parallel" if solver_cpus > 1 else "serial",
    }

    sol = smesolve(
        H_free,
        rho0,
        times,
        c_ops=collapsing_operators(gamma, phi1, phi2, 1.0 - eta_1, 1.0 - eta_2),
        sc_ops=collapsing_operators(gamma, phi1, phi2, eta_1, eta_2),
        heterodyne=False,
        e_ops=e_ops,
        ntraj=int(ntraj),
        options=options,
        seeds=seed,
    )
    return sol.expect


def simulate_single_eta(
    eta_1: float,
    eta_2: float,
    rho0,
    times: np.ndarray,
    gamma: float,
    phi1: float,
    phi2: float,
    e_ops: list,
    columns: list[str],
    ntraj: int,
    num_cpus: int,
    seed: int | None,
):
    avg_expect = run_homodyne(
        rho0=rho0,
        times=times,
        gamma=gamma,
        phi1=phi1,
        phi2=phi2,
        eta_1=float(eta_1),
        eta_2=float(eta_2),
        e_ops=e_ops,
        ntraj=int(ntraj),
        num_cpus=int(num_cpus),
        seed=seed,
    )
    key = eta_pair_key(float(eta_1), float(eta_2))
    return key, pd.DataFrame(np.transpose(avg_expect), columns=columns)

## Configuration

In [ ]:
cfg = {
    "state": "plusplus",
    "theta": float(np.pi / 2.0),
    "phi_state": 0.0,
    "gamma": 1.0,
    "phi1": 0.0,
    "phi2": 0.0,
    # Legacy-comparable preset: eta_1 and eta_2 move together.
    "etas_1": "1,0.9,0.7,0.5,0.3",
    "etas_2": "1,0.9,0.7,0.5,0.3",
    "ntraj": 100,
    "t_end": 10.0,
    "dt": 0.01,
    "num_cpus": 1,
    "seed": 12345,
    "no_theory": False,
    "usetex": False,
    "save_outputs": False,
    "out_root": r".\Codes\Graphs\Jz_2_Homodyne_N2",
}
cfg

In [ ]:
# Build state and run all eta pairs (serial for transparent debugging)
etas = parse_etas(cfg["etas_1"], cfg["etas_2"])
psi0, theta_for_theory = build_initial_state(cfg["state"], cfg["theta"], cfg["phi_state"])
rho0 = ket2dm(psi0)

gamma = float(cfg["gamma"])
t1 = 1.0 / gamma
times = np.arange(0.0, float(cfg["t_end"]) * t1, float(cfg["dt"]) * t1)
columns = [name for name, _ in DEFAULT_EOPS]
e_ops = [op for _, op in DEFAULT_EOPS]

results = []
for idx, (eta_1, eta_2) in enumerate(etas):
    eta_seed = int(cfg["seed"]) + 1000 * idx
    key, df = simulate_single_eta(
        eta_1=float(eta_1),
        eta_2=float(eta_2),
        rho0=rho0,
        times=times,
        gamma=gamma,
        phi1=float(cfg["phi1"]),
        phi2=float(cfg["phi2"]),
        e_ops=e_ops,
        columns=columns,
        ntraj=int(cfg["ntraj"]),
        num_cpus=int(cfg["num_cpus"]),
        seed=eta_seed,
    )
    results.append((key, df))

ineff_df = {k: v for k, v in results}
list(ineff_df.keys())

In [ ]:
# Quick numeric table at final time
summary_rows = []
for eta_1, eta_2 in etas:
    key = eta_pair_key(float(eta_1), float(eta_2))
    row = {"eta_1": eta_1, "eta_2": eta_2}
    for col in columns:
        row[f"{col}_final"] = float(ineff_df[key][col].to_numpy()[-1])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# Optional comparison with legacy NPZ output
legacy_npz = Path(r"Codes/Graphs/Jz_2_Homodyne_N2/phi_1=0_phi_2=0/homodyne_phi1=0_phi2=0.npz")

if legacy_npz.exists():
    old = np.load(legacy_npz)
    old_labels = [str(x) for x in old["labels"]]
    print("legacy labels:", old_labels)

    compare_rows = []
    for lab in old_labels:
        eta = float(lab)
        key = eta_pair_key(eta, eta)
        if key not in ineff_df:
            continue

        new_curve = ineff_df[key]["Conc"].to_numpy(dtype=float)
        old_curve = old["Conc"][old_labels.index(lab)].astype(float)

        n = min(len(new_curve), len(old_curve))
        max_abs = float(np.max(np.abs(new_curve[:n] - old_curve[:n])))
        compare_rows.append({"eta": eta, "points_compared": n, "max_abs_diff_Conc": max_abs})

    pd.DataFrame(compare_rows)
else:
    print(f"Legacy file not found: {legacy_npz}")

In [ ]:
# Reproducibility check: same seed must give identical curve
eta_1, eta_2 = etas[0]
seed_test = 999

_, df_a = simulate_single_eta(
    eta_1=float(eta_1),
    eta_2=float(eta_2),
    rho0=rho0,
    times=times,
    gamma=gamma,
    phi1=float(cfg["phi1"]),
    phi2=float(cfg["phi2"]),
    e_ops=e_ops,
    columns=columns,
    ntraj=int(cfg["ntraj"]),
    num_cpus=int(cfg["num_cpus"]),
    seed=seed_test,
)

_, df_b = simulate_single_eta(
    eta_1=float(eta_1),
    eta_2=float(eta_2),
    rho0=rho0,
    times=times,
    gamma=gamma,
    phi1=float(cfg["phi1"]),
    phi2=float(cfg["phi2"]),
    e_ops=e_ops,
    columns=columns,
    ntraj=int(cfg["ntraj"]),
    num_cpus=int(cfg["num_cpus"]),
    seed=seed_test,
)

max_abs_diff = float(np.max(np.abs(df_a["Conc"].to_numpy() - df_b["Conc"].to_numpy())))
max_abs_diff

In [ ]:
# Plots
plt.rcParams["text.usetex"] = bool(cfg["usetex"])
plt.rcParams.update(
    {
        "mathtext.fontset": "cm",
        "font.family": "serif",
        "font.size": 13,
        "axes.unicode_minus": False,
    }
)

x = times / t1

for col in columns:
    plt.figure(figsize=(11, 7))
    for eta_1, eta_2 in etas:
        key = eta_pair_key(float(eta_1), float(eta_2))
        y = ineff_df[key][col].to_numpy()
        line, = plt.plot(x, y, label=rf"$\eta_1={eta_1:g},\ \eta_2={eta_2:g}$")

        if col == "Conc" and (not cfg["no_theory"]) and np.isclose(eta_1, eta_2):
            y_th = theoretical_concurrence_curve(times=times, gamma=gamma, eta=float(eta_1), theta=theta_for_theory)
            plt.plot(x, y_th, "--", color=line.get_color(), alpha=0.55, linewidth=2)

    plt.xlim(0.0, float(cfg["t_end"]))
    plt.xlabel(r"$t/T_1$")
    plt.ylabel(LABELS[col])
    plt.title(r"Homodyne $J_z$ (N=2): " + LABELS[col])
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), fontsize=10)
    plt.show()

In [ ]:
# Optional: save outputs in the same layout used by the .py script
if cfg["save_outputs"]:
    phi1_dir = angle_to_path(float(cfg["phi1"]))
    phi2_dir = angle_to_path(float(cfg["phi2"]))
    out_dir = Path(cfg["out_root"]) / f"phi_1={phi1_dir}_phi_2={phi2_dir}"
    out_dir.mkdir(parents=True, exist_ok=True)

    for col in columns:
        plt.figure(figsize=(11, 7))
        for eta_1, eta_2 in etas:
            key = eta_pair_key(float(eta_1), float(eta_2))
            y = ineff_df[key][col].to_numpy()
            plt.plot(x, y, label=rf"$\eta_1={eta_1:g},\ \eta_2={eta_2:g}$")

        plt.xlim(0.0, float(cfg["t_end"]))
        plt.xlabel(r"$t/T_1$")
        plt.ylabel(LABELS[col])
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), fontsize=10)
        plt.savefig(out_dir / f"Homodyne_{col}.pdf", bbox_inches="tight")
        plt.close()

    meta = {
        "measurement": "homodyne",
        "N": 2,
        "state": cfg["state"],
        "theta": float(cfg["theta"]),
        "phi_state": float(cfg["phi_state"]),
        "gamma": float(cfg["gamma"]),
        "phi1": float(cfg["phi1"]),
        "phi2": float(cfg["phi2"]),
        "etas_all": [[float(a), float(b)] for a, b in etas],
        "ntraj": int(cfg["ntraj"]),
        "num_cpus_requested": int(cfg["num_cpus"]),
        "dt_T1": float(cfg["dt"]),
        "t_end_T1": float(cfg["t_end"]),
        "columns": columns,
        "seed": int(cfg["seed"]),
    }

    run_config_path = out_dir / "run_config.json"
    run_config_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Saved in: {out_dir}")
else:
    print("save_outputs=False -> no files written")